# 01 – Deploy Infrastructure

This notebook deploys Azure resources using the Bicep template in `infra/`, then **automatically writes the endpoint and credentials to `env/.env`** so subsequent notebooks can load them without manual copy-paste.

**Estimated time:** 10–20 minutes

---

## What gets deployed

<!-- List the Azure resources created by infra/main.bicep -->
- _Resource 1_
- _Resource 2_

---

## Step 1 – Set deployment variables

In [ ]:
RESOURCE_GROUP = "rg-<demo-name>-feasibility"   # <- change if needed
LOCATION = "swedencentral"                  # <- change if needed
INFRA_TEMPLATE_KIND = "bicep"               # options: "bicep" or "arm"

# Add demo-specific parameters here, e.g.:
# ACCOUNT_NAME = "<unique-account-name>"

template_file_map = {
    "bicep": "../infra/main.bicep",
    "arm": "../infra/main.json",
}

template_kind = INFRA_TEMPLATE_KIND.lower().strip()
if template_kind not in template_file_map:
    raise ValueError(f"Unsupported INFRA_TEMPLATE_KIND: {INFRA_TEMPLATE_KIND}. Use 'bicep' or 'arm'.")

TEMPLATE_FILE = template_file_map[template_kind]

print(f"Resource group : {RESOURCE_GROUP}")
print(f"Location       : {LOCATION}")
print(f"Template kind  : {template_kind}")
print(f"Template file  : {TEMPLATE_FILE}")

## Step 2 – Create the resource group

In [ ]:
!az group create \
    --name {RESOURCE_GROUP} \
    --location {LOCATION} \
    --output table

## Step 3 - Deploy infrastructure template

In [ ]:
import subprocess
import sys
from pathlib import Path

demo_root = Path("..").resolve()
if str(demo_root) not in sys.path:
    sys.path.insert(0, str(demo_root))

# Import helper if available; otherwise define resolve_az_cli locally
try:
    from app.notebook_helpers import resolve_az_cli
except ImportError:
    def resolve_az_cli():
        import shutil
        az_in_path = shutil.which("az")
        if az_in_path:
            return az_in_path
        windows_fallbacks = [
            Path(r"C:\\Program Files\\Microsoft SDKs\\Azure\\CLI2\\wbin\\az.cmd"),
            Path(r"C:\\Program Files (x86)\\Microsoft SDKs\\Azure\\CLI2\\wbin\\az.cmd"),
        ]
        for candidate in windows_fallbacks:
            if candidate.exists():
                return str(candidate)
        return None

az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError("Azure CLI not found. Install it or add it to PATH.")

template_path = Path(TEMPLATE_FILE)
if not template_path.exists():
    raise FileNotFoundError(f"Template file not found: {template_path.resolve()}")

deploy_result = subprocess.run(
    [
        az_cmd,
        "deployment",
        "group",
        "create",
        "--resource-group",
        RESOURCE_GROUP,
        "--template-file",
        str(template_path),
        "--output",
        "json",
        # Add --parameters key=value as needed
    ],
    capture_output=True,
    text=True,
    check=False,
)

if deploy_result.returncode != 0:
    print(f"Deployment failed for {template_kind} template:")
    print(f"\nstderr:\n{deploy_result.stderr}")
    print(f"\nstdout:\n{deploy_result.stdout}")
    raise RuntimeError(f"Deployment failed with exit code {deploy_result.returncode}")

print(f"Deployment succeeded using {template_kind} template")

## Step 4 – Retrieve outputs from the deployed resources

Query Azure for the endpoint and key. Adapt the `az` commands below to match your resource types.

In [ ]:
import subprocess
from pathlib import Path

def resolve_az_cli():
    import shutil
    az_in_path = shutil.which("az")
    if az_in_path:
        return az_in_path
    windows_fallbacks = [
        Path(r"C:\\Program Files\\Microsoft SDKs\\Azure\\CLI2\\wbin\\az.cmd"),
        Path(r"C:\\Program Files (x86)\\Microsoft SDKs\\Azure\\CLI2\\wbin\\az.cmd"),
    ]
    for candidate in windows_fallbacks:
        if candidate.exists():
            return str(candidate)
    return None

az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError("Azure CLI not found from this kernel. Install it or add it to PATH.")

print(f"Using Azure CLI: {az_cmd}")

# Example: retrieve an endpoint and key for a Cognitive Services account.
# Replace with the az commands that match your deployed resources.

# endpoint_result = subprocess.run(
#     [az_cmd, "cognitiveservices", "account", "show",
#      "--name", ACCOUNT_NAME,
#      "--resource-group", RESOURCE_GROUP,
#      "--query", "properties.endpoint",
#      "--output", "tsv"],
#     capture_output=True, text=True
# )
# endpoint = endpoint_result.stdout.strip()

# key_result = subprocess.run(
#     [az_cmd, "cognitiveservices", "account", "keys", "list",
#      "--name", ACCOUNT_NAME,
#      "--resource-group", RESOURCE_GROUP,
#      "--query", "key1",
#      "--output", "tsv"],
#     capture_output=True, text=True
# )
# api_key = key_result.stdout.strip()

# print(f"Endpoint : {endpoint}")
# print(f"Key      : {api_key[:8]}... (truncated)")
print("TODO: replace commented code with commands for your resource type")

## Step 5 – Write outputs to `env/.env`

This populates the `.env` file automatically from the deployed resource values.

> **Security note:** `env/.env` is git-ignored. Never commit it.

In [ ]:
import pathlib
import subprocess
from pathlib import Path

def resolve_az_cli():
    import shutil
    az_in_path = shutil.which("az")
    if az_in_path:
        return az_in_path
    windows_fallbacks = [
        Path(r"C:\\Program Files\\Microsoft SDKs\\Azure\\CLI2\\wbin\\az.cmd"),
        Path(r"C:\\Program Files (x86)\\Microsoft SDKs\\Azure\\CLI2\\wbin\\az.cmd"),
    ]
    for candidate in windows_fallbacks:
        if candidate.exists():
            return str(candidate)
    return None

az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError("Azure CLI not found from this kernel. Install it or add it to PATH.")

# Build the .env content from the values retrieved in Step 4.
# Add or remove lines to match your demo's env/.env.example.
env_content = f"""# Auto-generated by 01_deploy_infra.ipynb - do not commit
# SOME_ENDPOINT={endpoint}
# SOME_KEY={api_key}
"""

env_path = pathlib.Path("../env/.env")
env_path.write_text(env_content, encoding="utf-8")
print(f"✅  Written to {env_path.resolve()}")

## Step 6 - Verify `env/.env`

Confirm the generated environment file exists and contains expected keys before moving on.

In [ ]:
import pathlib

env_path = pathlib.Path("../env/.env")
example_path = pathlib.Path("../env/.env.example")

if env_path.exists():
    print("env/.env found")
    print(f"Path: {env_path.resolve()}")
else:
    print("ERROR: env/.env not found")
    print("Step 5 should generate this file.")
    print(f"Expected variable contract from {example_path}:")
    print(example_path.read_text(encoding="utf-8"))
    raise FileNotFoundError(env_path)

---

## ✅ Infrastructure deployed

Move to **`02_use_case.ipynb`** to exercise your demo scenario.

---

## Tear-down

In [ ]:
# Uncomment and run to delete all resources when finished.
# !az group delete --name {RESOURCE_GROUP} --yes --no-wait
# print("Resource group deletion triggered.")